# 第1回 演習：データの探索的分析

## 今日の分析目標

**自転車シェアの利用台数は何で決まりそうか、見当をつける。**

正解を求めるのではなく、pandasでデータを読み込み、形・統計量・図から「利用台数と関係がありそうな要因」や「注意が必要そうな列」を自分の言葉で言えるようになることが目標。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。

## 学習ゴール

この回を終えると、次のことができるようになります。

- `describe()` や欠損値（missing value）の確認から、データの**形と癖**（似た変数、カテゴリ然とした数値など）を自分で言い当てられる
- 目的変数（response variable）の分布をヒストグラム・箱ひげ図で確認し、「的」がどんな形をしているかを説明できる
- 散布図と相関行列（correlation matrix）を使い分け、目的変数と関係が強そうな変数の**手がかり**を挙げられる
- 相関はあくまで下見であり因果の証明ではないと、交絡（季節など）を例に説明できる


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
import japanize_matplotlib

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')

### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は色の濃いところを扱います。


In [ ]:
# 図：分析の地図（全14回のどこにいるか）
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
here = {1, 2, 3}          # ← 回ごとに変える（下の表を参照）
for i, (p, c, x) in enumerate(zip(phases, colors, [0.17, 0.5, 0.83]), start=1):
    on = i in here
    ax.text(x, 0.62, p, ha='center', va='center', fontsize=13 if on else 11,
            color='white', bbox=dict(boxstyle='round,pad=0.6', facecolor=c,
                                     alpha=0.95 if on else 0.25))
for x0, x1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(x1, 0.62), xytext=(x0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


## 1. データの読み込み

### 深掘り：この演習が向かう先——「予測」と「説明」、そして回帰の式

これから数回にわたって取り組む題材は**回帰**、つまり「手がかりの列（説明変数）から、当てたい値（目的変数）を推し量る」分析です。ここでの目的変数は `cnt`（その日の利用台数）、残りの列がすべて手がかりの候補になります。この節でわざわざ `casual`（一般客）と `registered`（会員）を落としているのには理由があります。この2つは足すとちょうど `cnt` に一致する内訳（`casual + registered = cnt`）なので、残したままにすると相関はぴったり 1 になり、「何が利用台数を決めるか」という問いの答えが自明になってしまいます。答えを先に配ってしまわないよう伏せ、天気・気温・季節といった**当日の状況**から正直に推し量る土俵に整えているわけです。

**「予測」と「説明」は別の狙い**　同じ回帰でも、力点の置き方で2つの目標に分かれます。

- **予測（predict）**：まだ見ぬ日の `cnt` を、できるだけ正確に**当てる**こと。極端に言えば中身がブラックボックスでも、よく当たれば役に立ちます。
- **説明（explain）**：どの要因が、どちら向きに、どれだけ効くのかを**読み取り、人に語る**こと。ここでは当たること以上に、係数が素直に読めて安定していることが大事になります。

この2つは両立することも、ぶつかることもあります。この演習シリーズは両方を行き来します——モデルを立てて手がかりを式にし（説明の芽）、その式がどれだけ当たるか（予測の精度）を測り、係数をどう読むか（説明）まで進みます。「今は予測と説明のどちらを問うているのか」を意識すると、各回の狙いが見通せます。

**回帰の予測式（意味だけ先取り）**　回帰が最終的に作るのは、次のような**予測の式**です。

$$
\hat y \;=\; \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p
\;=\; \beta_0 + \sum_{j=1}^{p} \beta_j x_j
$$

記号をほどきます。$\hat y$（ワイ・ハット）は式が出す**予測された利用台数**。$x_1,\dots,x_p$ はその日の手がかり（気温・湿度・季節など）で、$p$ は手がかりの本数です。$\beta_0$ は**切片**——すべての手がかりが基準の値のときの**土台の台数**にあたります。そして各 $\beta_j$ が肝で、「**手がかり $x_j$ が1単位増えると、予測 $\hat y$ が何台動くか**」という、その要因の**効きの大きさと向き**を表します。$\beta_j$ がプラスなら押し上げ、マイナスなら押し下げ。今日これから測る相関は、この $\beta_j$ たちの**符号と強さの下見**にあたります。

今日はまだ式そのものは立てません。まず的（`cnt`）と手がかり（各列）の顔ぶれと関係を知るのが仕事です。係数 $\beta_j$ を「予測の外れが最小になるように」機械的に決める手続き（最小二乗）は第2回の演習で自分の手で組み立てます。その手続きを、変数がいくつあっても通用する行列の形で厳密に導出するところは、回帰分析を体系立てて扱う統計学の教科書に譲ります。


In [ ]:
# 自転車シェアの日次データ（1日1行）を読み込む
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/bike_day.csv')

# instant(通し番号)・dteday(日付)・casual/registered は今回は使わないので落とす
#   ※ casual + registered = cnt（利用台数）なので、残すと相関の答えが自明になってしまう
df = df.drop(columns=['instant', 'dteday', 'casual', 'registered'])

print(f'行数: {df.shape[0]}, 列数: {df.shape[1]}')
print('目的変数: cnt（その日の利用台数）')
df.head()

## 2. 基本統計量

### 深掘り：describe をどう読むか——そっくりな双子の伏線

`describe()` は各列を1本の要約に凝縮した表です。行の意味を押さえておきましょう。**count** は件数、**mean** は平均（分布の中心の目安）、**std** は標準偏差（ばらつき＝典型的なズレ幅で、大きいほど日ごとの差が激しい）。**min / 25% / 50% / 75% / max** は、値を小さい順に並べたときの位置（分位点）で、真ん中の 50% が**中央値**です。

的である `cnt` を読むと、平均 **4504 台**、標準偏差（standard deviation） **1937 台**。ざっくり「平均±ばらつき」で 2500〜6400 台くらいが典型で、最小 22・最大 8714 と日によって大きく振れます。ここで平均 4504 と中央値（50%）4548 がほぼ一致しているのは覚えておいてください——分布の節で「素直な形」の裏付けとして効いてきます。

**そっくりな双子 `temp` と `atemp`**　`temp`（気温）と `atemp`（体感温度）の行を見比べると、平均 0.495 / 0.474、標準偏差 0.183 / 0.163 と、**姿かたちがほとんど同じ**です。実際、2列の相関は **0.99**（TODO①の describe と、7節の相関行列で確認できます）。中身がほぼ重なった2列——いわば**双子**が並んでいます。

これは後で効いてくる伏線です。同じ情報を持つ2列を両方モデルに入れると、「気温の効き」を `temp` と `atemp` の**どちらの係数に振り分けるか**をデータが決めきれず、係数がぐらついて素直に読めなくなります（**多重共線性**）。前の深掘りで触れた $\beta_j$ の「効きの大きさ」が、双子のあいだで宙ぶらりんになるわけです。今は「そっくりな列がある」と**気づくだけ**で十分。その正体と対処（片方を落とす、まとめる、正則化で手綱を締める）は、第6回・第7回の演習でじっくり扱います。


In [ ]:
df.describe().round(2)

### TODO①：気になる列を挙げてみよう

上の `df.describe()` の出力を見て、「あれ？」と思う列や、性質の似ている列を1つ以上挙げ、理由をコメントに書いてください。コードで何かを計算する必要はありません。

In [ ]:
# TODO: describe() の出力を見て、気になる列とその理由をコメントで書いてください
# ヒント1: temp（気温）と atemp（体感温度）の平均・ばらつきを見比べてみましょう
# ヒント2: season や weathersit は数値ですが、その大小に意味はあるでしょうか
...

## 3. 欠損値の確認

In [ ]:
print('欠損値の数:')
print(df.isnull().sum())

## 4. 目的変数（利用台数）の分布

### 深掘り：なぜ最初に「的」の分布を見るのか——ヒストグラムと箱ひげ図

目的変数は予測の**的**です。的がどのあたりに集まり、どこまで散らばり、いびつな形をしていないか——それを知らずにモデルは作れません。数字の要約（`describe`）だけでは分布の**形**までは見えないので、図で確かめます。

**ヒストグラムの読み方**　横軸が利用台数、縦軸がその台数だった日数です。`cnt` は 4000〜5000 台あたりが多く、低い日から高い日まで**なだらかに**広がっています。大きな歪みもありません。歪みを数字で見る **skew（歪度）** は約 **-0.05** と、ほぼ 0（＝左右対称）。基本統計量で見た平均 4504 と中央値 4548 がほぼ一致していたのも、この対称性の裏返しです。参考までに、所得や住宅価格のような分布は右に長い裾を引き、**平均 > 中央値**と大きくずれます。`cnt` はそうならないので、**平均が分布の中心の良い代表**になっています。

**箱ひげ図の読み方と「外れ値（outlier）」の直感**　箱は下から Q1（25%）〜Q3（75%）で、真ん中の 50% が入る範囲。箱の中の線が中央値です。箱の上下に伸びる**ひげ**は、おおむね箱の端から `1.5 × IQR`（IQR = Q3 − Q1）までの範囲を示し、**ひげの外にはみ出た点が「外れ値の候補」**として描かれます。このデータで境界を計算すると、Q1=3152・Q3=5956・IQR=2804 なので、フェンスは下側 **−1054**／上側 **10162**。`cnt` の実際の最小 22・最大 8714 はこの内側にすべて収まり、**箱ひげ図には外れ値の点が1つも立ちません**。極端に外れた日がなく、素直な分布だと確認できます。

ただし「自動ルールが何も出さない＝安心して放置してよい」ではない、という感覚も持っておきましょう。最小の 22 台は、典型の 4500 台前後と比べれば**異様に静かな日**です。それでもフェンスの内側に収まるのは、全体のばらつき（IQR）がそもそも大きいから。1.5×IQR は機械的な目安にすぎず、**目で見て「これは変だ」と引っかかった点は、ルールが黙っていても一度疑う**——それが外れ値との正しい付き合い方です。実務では的の分布が右に歪んだり外れ値だらけだったりも普通で、そのときは対数変換や外れ値処理を検討します（前処理の回のテーマ）。

**なぜ回帰に効くのか（軽く）**　次回以降の回帰は、多くの場合「予測の外れを二乗して足した合計」を最小にするように係数を決めます。二乗するぶん、極端に外れた1点が線を強く引っぱります。的の分布が今回のように素直だと、その心配が小さい——だから最初に的の形を見ておくのです。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['cnt'], bins=40, color='#0066cc', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('1日の利用台数 cnt')
axes[0].set_title('ヒストグラム')
axes[1].boxplot(df['cnt'])
axes[1].set_title('箱ひげ図')
plt.tight_layout()
plt.show()

## 5. 説明変数の分布

### 深掘り：数値の顔をしたカテゴリと、0〜1に揃えられた気温

説明変数（explanatory variable）のヒストグラムを並べると、形が**2種類**あることに気づきます。`temp`・`atemp`・`hum`・`windspeed` はなめらかに散らばる**連続値**。いっぽう `season`（1〜4の4本）・`weathersit`（1〜3の3本）・`yr`（0/1）などは、**数本の棒しか立たない離散値**です。この違いが、後の前処理で効いてきます。

**数値の顔をしたカテゴリ**　`season` の 1,2,3,4 や `weathersit` の 1,2,3 は数字で入っていますが、その**大小や間隔には意味がありません**。`season=4` が `season=1` の「4倍」ではなく、ただの区分（春夏秋冬）につけた番号です。`weathersit` は 1=晴れ〜3=悪天候という**順序**こそありますが、「3 は 1 の3倍悪い」わけではありません。

**なぜ困るのか**　この番号のまま回帰に入れると、係数は「`season` が1増えると `cnt` が $\beta$ 台動く」という、**等間隔で一直線な効き**を勝手に仮定してしまいます。ところが季節の効きは直線ではありません。実際、季節ごとの平均利用台数は 区分1 **2604** → 区分2 **4992** → 区分3 **5644** → 区分4 **4728** と、**いったん上がってまた下がる山型**で、番号の順にまっすぐ増えてはいません（そもそも番号は寒暖の順ですらなく、最も暖かいのは区分3です）。番号のまま突っ込むと、この山型の効きを直線と取り違えてしまうのです。

**対処の伏線（ワンホット）**　これを避けるには、カテゴリを「その区分か否か」の 0/1 の列に開きます。`season` の1列を「春か」「夏か」「秋か」「冬か」という複数の 0/1 列に分ければ、各季節の効きを**個別に**測れて、山型でも正しく捉えられます。これが**ワンホット（one-hot）表現**で、第4回の演習（前処理）で自分の手で作ります。

**別の見方：0〜1に揃えられた気温**　`temp`・`atemp` が 0〜1 に収まっているのは、元の気温を取りうる範囲で割って**正規化（normalization）**した値だからです。だから `describe` の平均 0.5 付近は「摂氏◯度」ではなく「範囲の真ん中あたり」という意味。こうして単位を揃えておく操作は、後で複数の変数の係数を**公平に比べる**ときに効いてきます（第7回の演習で、平均0・ばらつき1に揃える**標準化**として再登場します）。


In [ ]:
features = ['temp', 'atemp', 'hum', 'windspeed', 'season', 'weathersit', 'mnth', 'yr']
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.flat, features):
    ax.hist(df[col], bins=30, color='#2a9d8f', alpha=0.8, edgecolor='white')
    ax.set_title(col, fontsize=10)
plt.suptitle('説明変数の分布', fontsize=13)
plt.tight_layout()
plt.show()

## 6. 説明変数 vs 利用台数の散布図

### 深掘り：一つの数字に潰れる前の「関係の形」を見る

次の節では、変数どうしの関係を相関係数という**1つの数字**に要約します。便利ですが、数字1つでは**関係の形**までは分かりません。散布図は、その要約の手前にある**生の散らばり**を自分の目で見るための道具です。

**気温 vs 利用台数（TODO②）**　`temp` と `cnt` の点の雲は、全体として**右上がり**——暖かい日ほど利用が多い傾向がはっきり見えます。ところが右端、いちばん暑いあたりをよく見ると、伸びが止まって**むしろやや下がって**いるようにも見えます。暑すぎる日は、かえって自転車が敬遠されるのかもしれません。つまり関係は右上がりでも、**一直線とは限らない**のです。

**なぜ大事か**　相関係数が測るのは、あくまで「**直線的な**」一緒の動きです。上のように山なり・頭打ちの関係だと、相関係数は本当の関係の強さを**取りこぼす**（過小評価する）ことがあります。だから数字（相関）と図（散布図）を**必ず両方**見る——これが基本の型です。数字だけを信じると「直線でない効き」を見落とします。

**別の見方と、少し先の話**　散布図の点は、同じ気温でも縦に幅を持って散らばっています。この縦の散らばりこそ、気温だけでは説明しきれない**残りのばらつき**——年・季節・天気といった他の要因が効く余地です。1変数の散布図は関係の入口にすぎず、本命は複数の要因を**同時に**見る回帰（第2回以降）になります。もし直線では足りないと分かれば、後の回で気温の2乗の項を足したり非線形（nonlinear）の手法に広げたりする拡張もあります。まずは「**形を見て、直線でよいかを疑う**」目を持つことが、この節の狙いです。


### TODO②：任意の変数を1つ選び、利用台数との散布図を描く

`features` の中から気になる変数を1つ選び、`cnt`（利用台数）との散布図を描いてください。点の散らばり方から、その変数と利用台数の関係について気づいたことを考えてみましょう。

In [ ]:
# TODO: features の中から好きな変数を1つ選び、cnt（利用台数）との散布図を描いてください
# ヒント: plt.scatter(df['変数名'], df['cnt'], alpha=0.3, s=8) のように書ける
...

## 7. 相関行列

### 深掘り：相関は手がかり、因果ではない——隠れた交絡としての季節

相関行列は、列どうしの相関係数（一緒に動く度合い、$-1$〜$1$）を一覧にした表です。`cnt` との相関を強い順に読むと、手がかりの**下見の順位**が出ます（TODO③で取り出すのがこれです）。上位は `atemp` **0.63**・`temp` **0.63**・`yr` **0.57**・`season` **0.41** と続き、逆向き（負）には `weathersit` **−0.30**・`windspeed` **−0.24**・`hum` **−0.10**。双子の `atemp` と `temp` が**ほぼ同点**で最上位に並ぶのは、2列がそっくりだから当然の結果です。

ここで、データ分析でいちばん有名な落とし穴に触れます。**相関が強い ≠ 因果**。「気温と利用台数の相関が 0.63」は、「気温が高いことが**原因で**利用が増える」と言い切れるわけではありません。

**隠れた交絡（confounder）としての季節**　`season`（季節）は、気温と利用台数の**両方を同時に**動かす背後の要因です。暖かい季節は気温が高く（区分別の平均 `temp`：最も寒い区分1で 0.30、最も暖かい区分3で 0.71）、その暖かい季節はお出かけも増えて利用も多い（同じく区分別の平均 `cnt`：区分1で 2604、区分3で 5644）。すると「気温が高い日に利用が多い」という相関の**一部は、実は季節という共通原因が作ったもの**かもしれません。気温そのものを動かさなくても、季節が動けば両方が動くからです。こういう背後の共通原因を**交絡**と呼びます。

**実データで一目**　全体では `temp`↔`cnt` の相関は **0.63**。ところが**季節ごとに区切って**その中で相関を測ると、様子が変わります——区分1で 0.67／区分2で 0.48／区分3で **−0.03**／区分4で 0.40。とりわけ最も暖かい区分3（夏にあたる）の中では、その日の気温の高低が利用台数を**ほとんど説明しません**（相関ほぼ 0）。「暑い季節どうし」で比べると気温の効きが消えるのは、その日の気温そのものより「どの季節か」が効いていた——つまり相関が**季節という文脈に支えられていた**ことを示します。前の節で見た「暑すぎる日は頭打ち」とも符合する結果です。文脈（どの季節か）を揃えると関係の強さがこれほど変わる以上、全体の一つの相関 0.63 を額面どおり「気温の因果」と読むのは早すぎるのです。

だから相関は、**どの要因を疑うべきかの手がかり**（下見）であって、因果の証明ではありません。因果に近づくには、交絡を揃える——同じ季節の中で比べる、あるいは季節を式に**一緒に**入れて「他を揃えたうえでの気温の効き」を見る——といった、さらに慎重な検討が要ります。複数の変数を同時に入れる回帰は、まさにその一歩で、第2回以降で手を動かします。交絡を調整した効きの測り方——他の変数を揃えたときの係数（偏回帰係数）がどう定義されるかは、計量経済学や統計学の教科書に譲ります。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, ax=ax, annot_kws={'size': 8})
ax.set_title('相関行列')
plt.tight_layout()
plt.show()

### TODO③：利用台数との相関 上位3変数を取り出す

上のヒートマップを参考に、`cnt`（利用台数）との相関係数を計算し、上位3変数をコードで取り出してください。

In [ ]:
# TODO: df.corr()['cnt'] を使って、利用台数との相関が高い上位3変数を取り出してください
# ヒント: sort_values(ascending=False) と .drop('cnt') が使える
...

## 目標に答えられたか

- 今日の目標は「自転車シェアの利用台数は何で決まりそうか、見当をつける」ことでした
- TODO③の結果を見て、利用台数と最も関係が強そうな変数は何だったでしょうか？
- TODO①で気づいた「そっくりな2つの列」や「数値の顔をしたカテゴリ」は、今後の分析でどう扱うべきだと思いますか？
- 分布に偏りがある変数はありましたか？次回以降、どんな前処理が必要になりそうでしょうか？

## 今日の要点

- 目的変数（`cnt`）の分布や癖を先に見ておくと、後で立てる回帰の結果を素直に読める
- ヒストグラムは1変数の癖、散布図は2変数の関係の**形**を見る道具で、相関係数だけでは見落とす非線形の関係を教えてくれる
- 相関行列は「何を疑うべきか」の手がかり一覧であって、因果の証明ではない
- `casual`・`registered` のように内訳どうしの変数を残すと答えが自明になるため、意図的に外して土俵を整えた
- 気温 `temp` と体感温度 `atemp` のような似た変数（双子）はすでに今日の相関表に現れており、後の回で係数のぐらつきとして効いてくる伏線になっている
- 今日は「何が関係しそうか」の見当をつけただけで、手がかりから的をどれだけ言い当てられるかはまだ数字にしていない


## 次回へ

今日は的（`cnt`）と手がかり（各列）の顔ぶれを、統計量と図でひと通り確かめました。気温のように関係が強そうな変数の見当はつきましたが、「手がかりから的をどれだけ言い当てられるか」はまだ数字にしていません。

次回は、いちばん相性の良さそうな気温を使って、実際に直線の式を立てます。良い直線を残差（residual）の二乗和という物差しで選ぶという、回帰の中身そのものを自分の手で組み立てましょう。


## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

1節で読み込んだ `bike_day.csv` は1日1行のデータでした。同じ自転車シェアには、1時間1行のデータ `bike_hour.csv` もあります（17,379行。列は bike_day と同じで、時刻を表す `hr`（0〜23）が加わります）。

`pd.read_csv(f'{DATA_DIR}/bike_hour.csv')` で時間別データを読み込み（`df` は後で使うので上書きせず、`hour` など別の名前にしてください）、時刻 `hr` ごとの利用台数 `cnt` の平均を求めて、横軸を時刻・縦軸を平均台数にした折れ線グラフを描いてください。集計には `groupby`（初出）を使います。使い方は「詰まったら」にあります。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

「時刻ごとの平均」は `hour.groupby('hr')['cnt'].mean()` で求まります（時刻を添字にした Series になります）。
その Series に `.plot(marker='o')` をつけると折れ線グラフになります。

</details>


### 応用②（判断）

応用①の結果で、平均利用台数が最も多いのは何時ですか。時刻を **0〜23 の整数** で答えてください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

Series の `.idxmax()` は、最大値の**位置**（ここでは時刻）を返します。値そのものは `.max()` です。

</details>


### 応用③（解釈）

応用①のグラフを根拠に、日次データ（bike_day）だけを見ていたときには見えなかった、**利用の時間帯パターンとそこから考えられること**を、自転車シェアの**運営担当者**に向けて3行程度で書いてください。


（ここに3行程度で書く）


## 発展（任意）

### 自動EDAレポート（data-profiling）

今日は `describe()`・ヒストグラム・相関行列を1つずつ自分で出しました。列が12本ならこれで十分ですが、実務のデータは列が数十〜数百本あるのが普通で、1列ずつ眺めるのは現実的ではありません。

`data-profiling`（旧名 ydata-profiling / pandas-profiling）は、データフレームを渡すだけで、各列の統計量・分布・欠損・相関・「注意すべき点（Alerts）」をまとめた1本のレポートを作ってくれるライブラリです。初めて見るデータの**全体像を数分でつかむ**のに向いています。

ただし、あくまで「見当をつける」道具です。レポートは何が「変」かの候補を並べるだけで、それが本当に問題かどうかの結論は自分で出します。


In [ ]:
# Colab には入っていないので、なければインストールする（配布名は fg-data-profiling）
import importlib.util
if importlib.util.find_spec('data_profiling') is None and importlib.util.find_spec('ydata_profiling') is None:
    %pip install -q fg-data-profiling
# このライブラリは古い仕組み pkg_resources を使うので、無い環境（setuptools 81 以降）では補う
if importlib.util.find_spec('pkg_resources') is None:
    %pip install -q 'setuptools<81'


In [ ]:
# 新しい名前 data_profiling を優先し、旧名 ydata_profiling しかない環境でも動くようにする
try:
    from data_profiling import ProfileReport
except ImportError:
    from ydata_profiling import ProfileReport

# minimal=True で重い計算を省く。相関だけは見たいので、その計算だけ有効にする
prof = ProfileReport(df, title='bike_day の自動EDA', minimal=True,
                     correlations={'auto': {'calculate': True}}, progress_bar=False)
prof.to_notebook_iframe()


**読み方**　レポートの上部にある **Alerts** タブから見ます。`temp` と `atemp` に「High correlation」の警告が出ます。TODO①で自分の目で見つけた「そっくりな双子」を、道具も同じように拾ってきたわけです。

一方で `yr`・`holiday`・`weekday`・`workingday` には「Zeros」（ゼロが多い）の警告が出ます。これらは 0/1 や曜日番号の列なので、ゼロが多いのは当たり前で、問題ではありません。道具は機械的に候補を挙げるだけなので、こうした**空振り**を自分で見分けるのが人間の仕事です。

**Variables** の項では列ごとの分布と統計量、**Correlations** の項では7節のヒートマップに相当する相関行列（列の型に応じて指標が自動選択されるため、数値は7節と少し異なります）が見られます。Colab で重いと感じたら、`prof.to_file('report.html')` でファイルに書き出し、ブラウザで開く方法もあります。

試すなら、応用①で読んだ時間別データ `hour` を渡してみてください。行数が24倍あるので、少し時間がかかります。
